# 11 — Save & Load Model with Pickle
Serialize the complete pipeline and verify prediction integrity.

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from src.config import MODEL_PATH, RANDOM_STATE
from src.inference import load_model, predict

In [ ]:
X_test = pd.read_csv('data/processed/X_test.csv')
y_test = pd.read_csv('data/processed/y_test.csv').values.ravel()

## 1. Why Pickle the ENTIRE Pipeline?
The pipeline contains both the preprocessor and the model.
If you pickle only the model, you lose the fitted scaler/encoder,
and inference on raw data will fail or produce wrong results.

## 2. Load Best Model from Previous Notebook

In [ ]:
# Re-train the best model (or load from previous notebook)
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from src.preprocess import build_preprocessor

X_train = pd.read_csv('data/processed/X_train.csv')
y_train = pd.read_csv('data/processed/y_train.csv').values.ravel()

dt_pipe = Pipeline([('preprocessor', build_preprocessor()),
                     ('classifier', DecisionTreeClassifier(random_state=RANDOM_STATE))])
grid_dt = GridSearchCV(dt_pipe, {'classifier__max_depth': [3, 5, 7, 10]}, cv=5, scoring='f1', n_jobs=-1)
grid_dt.fit(X_train, y_train)
final_pipeline = grid_dt.best_estimator_
print(f"Best model: {final_pipeline}")

## 3. Save to Pickle

In [ ]:
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)

with open(MODEL_PATH, 'wb') as f:
    pickle.dump(final_pipeline, f)

file_size = os.path.getsize(MODEL_PATH) / 1024
print(f"Model saved to {MODEL_PATH}")
print(f"File size: {file_size:.1f} KB")

## 4. Load and Verify

In [ ]:
loaded_model = load_model(MODEL_PATH)

# Compare predictions on 5 test rows
sample = X_test.iloc[:5]
preds_original = final_pipeline.predict(sample)
preds_loaded = loaded_model.predict(sample)

print(f"Original predictions: {preds_original}")
print(f"Loaded predictions:   {preds_loaded}")

assert np.array_equal(preds_original, preds_loaded), "PREDICTIONS DO NOT MATCH!"
print("\n✓ Predictions match. Pipeline integrity confirmed.")

## 5. Test inference.predict() function

In [ ]:
sample_row = X_test.iloc[[0]]
result = predict(loaded_model, sample_row)
print(f"Prediction: {result['prediction']} (0=No Default, 1=Default)")
print(f"Probability of default: {result['probability']:.4f}")